### Middleware

Middleware provides a way, to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behaviour with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
 - Long-running conversations that exceed context windows.
 - Multi turn dialogues with extensive history.
 - Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

### Message based Summarization
agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(), # Checkpoint is to save whatever conversation is there inside our hardware, in our memory
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [ ]:
### Run with Thread ID
config={"configurable":{"thread_id":"test-1"}}

In [ ]:
# Alternative Test Data
questions = [
    "What is 2+2?",
    "what is 10*5?",
    "What is 100/4",
    "What is 200*32",
    "What is 3*3*3*3",
    "What is 2* 1/2?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"messages: {response}"),
    print(f"messages: {len(response['messages'])}")

### Token Size

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym.
    2. City Inn - 4 star, $180/night, business center.
    3. Budget Stay - 3 star, $75/night, free wifi. """

agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model='groq:openai/gpt-oss-20b',
            trigger=("tokens", 550),
            keep=("tokens",200),
        ),
    ]
)
config={"configurable": {"thread_id": "test-1"}}

# Token Counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars roughly equals 1 token 

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens , {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

### Fraction

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool

@tool
def search_hotels(city: str):
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel 350$, City Inn 180$, Budget Stay 75$"

# Low fraction for testing

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "groq:openai/gpt-oss-20b",
            trigger = ("fraction", 0.005), # It takes the fraction of total output tokens
            keep = ("fraction", 0.002),
        ),
    ],
)

config = {"configurable": {"thread_id":"test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]
for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print (response['messages'])

### Human in the loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operation requiring human approval (eg. database writes, financial transactions)
- Compliance workflows where human oversight is mandatory
- Long-running conversations where human feedback guides the agent

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its id"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject:str, body:str) -> str:
    """Mock function to send an email."""
    return f"Email sent to '{recipient}' with the subject '{subject}'" 

In [30]:
agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1 Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='5489fda9-3099-4e6b-84e0-63783cde6171'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email_tool.', 'tool_calls': [{'id': 'fc_1236f147-1a64-414f-9cf7-e1f714369ce3', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 174, 'total_tokens': 221, 'completion_time': 0.052599853, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.010104973, 'prompt_tokens_details': None, 'queue_time': 0.3487223, 'total_time': 0.062704826}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_1074f9ce08', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, i

### For Approve

In [28]:
from langgraph.types import Command
# Step 2 Approval
if "__interrupt__" in result:
    print(" Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f" Result: {result['messages'][-1].content}")

 Paused! Approving...
 Result: ✅ Email sent to **john@test.com** with subject **“Hello”** and body **“How are you?”**. Let me know if you need anything else!


### For Reject

In [32]:
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.tools import tool

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools = [send_email_tool, read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "reject", "edit"]
                },
                "read_email_tool":False,
            }
        ),
    ],
)

In [35]:
config = {"configurable":{"thread_id" : "test-reject"}}
# Step 1 request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to test@john.com with Subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [37]:
# Step 2 - Reject
if "__interrupt__" in result:
    print(" Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f" Result: {result['messages'][-1].content}")

result

{'messages': [HumanMessage(content="Send email to test@john.com with Subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='82981b30-3391-4ce5-b582-e964a783ce51'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the send_email_tool with the specified arguments.', 'tool_calls': [{'id': 'fc_0ee231c2-dc06-416d-9ae8-bd3b0cc08b91', 'function': {'arguments': '{"body":"How are you?","recipient":"test@john.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 175, 'total_tokens': 227, 'completion_time': 0.069785664, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.059149053, 'prompt_tokens_details': None, 'queue_time': 0.056445075, 'total_time': 0.128934717}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

### For Editing

In [48]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email content"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject:str, body:str)-> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'."

agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools=[send_email_tool, read_email_tool],
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        ),
    ],
)

In [49]:
config = {"configurable":{"thread_id":"test-edit"}}
result = agent.invoke({
    "messages":[HumanMessage(content="Send an email to johndoe@test.com with subject 'Hi' and body 'how are you?'")]
    },
    config=config 
)
result

{'messages': [HumanMessage(content="Send an email to johndoe@test.com with subject 'Hi' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='239de143-8144-4a40-aff8-c7da4f7f02a8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants to send email to johndoe@test.com subject Hi body "how are you?" We have a tool send_email_tool. We need to call it.', 'tool_calls': [{'id': 'fc_919cc893-86a5-498e-9ab9-97202c7b7e51', 'function': {'arguments': '{"body":"how are you?","recipient":"johndoe@test.com","subject":"Hi"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 175, 'total_tokens': 249, 'completion_time': 0.085985223, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.014978664, 'prompt_tokens_details': None, 'queue_time': 0.354968625, 'total_time': 0.100963887}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_

In [54]:
# Step 2 - Edit
if "__interrupt__" in result:
    print("Paused! Editing...")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool", # Tool name
                            "args":{
                                "recipient":"right@test.com",
                                "subject":"Hey",
                                "body":"wazzzuupppp"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
print(f" Result: {result['messages'][-1].content}")

 Result: I’m sorry, but the email you requested wasn’t sent to the address you specified. The system rerouted the request and the email was sent to **right@test.com** with the subject **“Hey”** and body **“wazzzuupppp.”**

If you’d like me to send an email to **johndoe@test.com** with the subject **“Hi”** and body **“how are you?”**, please let me know and I’ll do it.
